# Parameter Recovery from Contrastive Embeddings

This notebook demonstrates how to train a recovery head that predicts the underlying ARMA parameters from the latent representations learned by the contrastive backbone.

**The idea:** If the contrastive model has learned meaningful representations of time series, its embeddings should encode information about the data-generating process. We test this by training a small head network to recover the AR and MA coefficients from the frozen backbone's latent features.

**Metrics:**
- **Improvement ratio**: how much better the model is vs. predicting zeros (baseline). Values >1 mean the model is useful.
- **Sign agreement**: fraction of coefficients where the predicted sign matches the true sign.

**Pre-trained backbone required.** This notebook loads a checkpoint. You can either:
1. Train one yourself using the `train_contrastive.ipynb` notebook (or `train_contrastive_v2.py`)
2. Use the pre-trained `trained_simple_model_H1024.pth` (411MB, trained for 2M steps on the original SimpleModel architecture)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from src.arma import generate_arma_batch
from src.models import ConfigurableModel
from src.recovery import create_recovery_head, parameter_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load the pre-trained backbone

Set `MODEL_PATH` to the checkpoint you want to use. The model architecture parameters must match the checkpoint.

If using the original `trained_simple_model_H1024.pth`, use the `SimpleModel` class instead of `ConfigurableModel`. Here we show the `ConfigurableModel` path (for checkpoints from `train_contrastive_v2.py`).

In [ ]:
C, H, W = 4, 1024, 32
num_arma_params = 4  # recover first 4 AR and 4 MA coefficients

# --- ConfigurableModel backbone (best architecture) ---
MODEL_PATH = "../checkpoints/model.pth"  # adjust to your checkpoint

backbone = ConfigurableModel(
    C=C, H=H, W=W,
    encoder_type='gru',
    num_layers=12,
    nhead=8,
    ffn_mult=4,
    activation='gelu',
    depthwise_conv=3,
)

# --- Alternative: original SimpleModel backbone (uncomment) ---
# from src.network import SimpleModel
# MODEL_PATH = "../checkpoints/trained_simple_model_H1024.pth"
# backbone = SimpleModel(C=C, H=H, W=W, num_layers=12)

backbone.load_state_dict(torch.load(MODEL_PATH, map_location=device))
backbone = backbone.to(device)
backbone.eval()
for param in backbone.parameters():
    param.requires_grad = False
print("Backbone loaded and frozen.")

## Create the recovery head

We use a GRU-based head that processes the temporal sequence of latent embeddings. The GRU aggregates information across time steps before predicting the (constant) ARMA parameters.

In [ ]:
recovery_head = create_recovery_head(
    'gru',
    H=H,
    num_arma_params=num_arma_params,
    hidden_dim=128,
    num_gru_layers=2,
).to(device)

n_params = sum(p.numel() for p in recovery_head.parameters() if p.requires_grad)
print(f"Recovery head parameters: {n_params:,}")

## Helper: extract features and prepare targets

In [ ]:
def extract_features(backbone, x):
    """Extract per-channel latent features from the frozen backbone."""
    with torch.no_grad():
        _, h = backbone(x)          # h: [B, T, C, H]
        B, T, C, H = h.shape
        return h.permute(0, 2, 1, 3).reshape(B * C, T, H)  # [B*C, T, H]


def prepare_batch(batch_size, seed=None):
    """Generate ARMA data and extract true parameters as targets."""
    x, parameters = generate_arma_batch(
        batch_size=batch_size, T_raw=4096, C=C, seed=seed, dimension=4
    )
    x = x.to(device)

    true_ar, true_ma = [], []
    for ar_poly, ma_poly in parameters:
        ar_coeffs = -ar_poly[1:]  # remove leading 1 and negate
        ma_coeffs = ma_poly[1:]   # remove leading 1
        ar_padded = np.pad(ar_coeffs, (0, max(0, num_arma_params - len(ar_coeffs))))[:num_arma_params]
        ma_padded = np.pad(ma_coeffs, (0, max(0, num_arma_params - len(ma_coeffs))))[:num_arma_params]
        true_ar.append(ar_padded)
        true_ma.append(ma_padded)

    true_ar = torch.tensor(np.array(true_ar), dtype=torch.float32).to(device)
    true_ma = torch.tensor(np.array(true_ma), dtype=torch.float32).to(device)
    return x, true_ar, true_ma

## Train the recovery head

We train for 5000 epochs. Each epoch generates a fresh batch of ARMA data, extracts backbone features, and optimizes the recovery head with MSE loss on the AR/MA coefficients.

In [ ]:
optimizer = optim.Adam(recovery_head.parameters(), lr=1e-3)

# Fixed validation set
x_val, ar_val, ma_val = prepare_batch(batch_size=32, seed=0)
h_val = extract_features(backbone, x_val)

total_epochs = 5000
best_val_loss = float('inf')

for epoch in range(1, total_epochs + 1):
    recovery_head.train()
    optimizer.zero_grad()

    x_train, ar_train, ma_train = prepare_batch(batch_size=32, seed=epoch)
    h_train = extract_features(backbone, x_train)

    pred_ar, pred_ma = recovery_head(h_train)
    loss, ar_loss, ma_loss = parameter_loss(pred_ar, pred_ma, ar_train, ma_train)
    loss.backward()
    optimizer.step()

    # Validation
    recovery_head.eval()
    with torch.no_grad():
        pred_ar_v, pred_ma_v = recovery_head(h_val)
        val_loss, _, _ = parameter_loss(pred_ar_v, pred_ma_v, ar_val, ma_val)

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_epoch = epoch

    if epoch % 500 == 0:
        print(f"Epoch {epoch:5d} | train={loss.item():.6f} | val={val_loss.item():.6f} | best={best_val_loss:.6f}@{best_epoch}")

## Evaluate: improvement ratio and sign agreement

We evaluate on 100 fresh test samples. The improvement ratio compares the model's MSE against the "predict zeros" baseline.

In [ ]:
recovery_head.eval()
num_test = 100

all_pred_ar, all_pred_ma = [], []
all_true_ar, all_true_ma = [], []
all_errors, all_baselines = [], []

with torch.no_grad():
    for i in range(num_test):
        x_test, ar_true, ma_true = prepare_batch(batch_size=1, seed=i + 10000)
        h_test = extract_features(backbone, x_test)

        pred_ar, pred_ma = recovery_head(h_test)
        _, ar_err, ma_err = parameter_loss(pred_ar, pred_ma, ar_true, ma_true)

        baseline = (F.mse_loss(torch.zeros_like(ar_true), ar_true) +
                    F.mse_loss(torch.zeros_like(ma_true), ma_true))

        all_errors.append(ar_err.item() + ma_err.item())
        all_baselines.append(baseline.item())

        all_pred_ar.append(pred_ar.mean(dim=1).cpu())
        all_pred_ma.append(pred_ma.mean(dim=1).cpu())
        all_true_ar.append(ar_true.cpu())
        all_true_ma.append(ma_true.cpu())

mean_error = np.mean(all_errors)
mean_baseline = np.mean(all_baselines)
improvement = mean_baseline / mean_error

# Sign agreement (ignoring zero-padded coefficients)
pred_ar_all = torch.cat(all_pred_ar).numpy()
pred_ma_all = torch.cat(all_pred_ma).numpy()
true_ar_all = torch.cat(all_true_ar).numpy()
true_ma_all = torch.cat(all_true_ma).numpy()

ar_nonzero = true_ar_all != 0
ma_nonzero = true_ma_all != 0
sign_ar = np.mean(np.sign(pred_ar_all[ar_nonzero]) == np.sign(true_ar_all[ar_nonzero]))
sign_ma = np.mean(np.sign(pred_ma_all[ma_nonzero]) == np.sign(true_ma_all[ma_nonzero]))

print(f"Mean MSE:          {mean_error:.6f}")
print(f"Baseline MSE:      {mean_baseline:.6f}")
print(f"Improvement ratio: {improvement:.2f}x")
print(f"Sign agreement AR: {sign_ar:.2%}")
print(f"Sign agreement MA: {sign_ma:.2%}")

## Plot: true vs. predicted parameters

Scatter plots of true vs. predicted values for each AR and MA coefficient. Points on the diagonal mean perfect recovery.

In [ ]:
fig, axes = plt.subplots(2, num_arma_params, figsize=(4 * num_arma_params, 8))

for j in range(num_arma_params):
    # AR coefficients
    ax = axes[0, j]
    ax.scatter(true_ar_all[:, j], pred_ar_all[:, j], alpha=0.3, s=10)
    ax.plot([-1, 1], [-1, 1], 'r--', linewidth=1)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(f"AR[{j}]")
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')

    # MA coefficients
    ax = axes[1, j]
    ax.scatter(true_ma_all[:, j], pred_ma_all[:, j], alpha=0.3, s=10)
    ax.plot([-1, 1], [-1, 1], 'r--', linewidth=1)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(f"MA[{j}]")
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')

fig.suptitle("True vs. Predicted ARMA Parameters", fontsize=14)
plt.tight_layout()
plt.show()